# EfficientViT B2 + forensic fusion, 1024×1024

Запускает `configs/efficientvit_b2_mixed_original.yaml`: EfficientViT MIT B2 (ImageNet r288), вход 1024×1024 с forensic-картами и настройками split/обучения из конфига.

Для нового эксперимента скопируйте YAML, измените параметры и путь ниже. Результаты: `runs/<run_name>/summary.json`, `metrics.csv`, `notes.md`; лучшие веса: `ckpt/best.pt`.


Полная модель: **97.008 GFLOPS**, 16.31 млн параметров. Форензика добавляется к encoder features на stride 8/16/32; U-Net decoder и обе головы сохранены.

Batch 2 × accumulation 8 = 16, бюджет 192 000 примеров, тот же fold и mixed/original протокол.

Pretrained-веса `timm/efficientvit_b2.r288_in1k` загружаются при первом запуске. Скорость на H100 и пиковая память обучения на RTX 3070 ещё не измерены. При нехватке памяти используйте batch 1 × accumulation 16 в копии YAML с новым run_name.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import load_experiment_config
from src.training.engine import run_experiment

cfg = load_experiment_config(project_root / "configs" / "efficientvit_b2_mixed_original.yaml")
cfg

ExperimentConfig(paths=PathsConfig(data_path=WindowsPath('D:/Challenges/AIIJC2026/data'), runs_path=WindowsPath('D:/Projects/aiijc2026_final/runs'), run_name='efficientvit_b2_1024_mixed_original'), seed=42, model=ModelConfig(encoder_name='efficientvit_b2.r288_in1k', decoder_channels=(128, 64, 32, 16, 16), forensic_channels=(64, 96, 128), aux_weight=0.4, norm='batch'), augmentation=AugmentationConfig(crop_scale_range=(0.35, 1.0), jpeg_recompression_probability=0.3, jpeg_recompression_quality_range=(60, 100), full_frame=False, full_frame_probability=0.5, foreground_crop_probability=0.5, final_full_frame_epochs=2), dataset=DatasetConfig(fold=0, n_folds=5, image_size=1024, resize_mode='stretch'), train=TrainConfig(device='cuda', workers=10, encoder_lr=0.0001, fmap_lr=0.0003, lr=0.0003, weight_decay=0.0001, epochs=8, epoch_size=24000, val_frac=0.25, val_keep_negatives=True, negative_fraction=0.25, batch_size=2, accum_steps=8, warmup_frac=0.05, min_lr_factor=0.02, amp='bf16', ema_decay=0.999

In [ ]:
run = run_experiment(cfg)
run.summary

[09:30:16] Эксперимент efficientvit_b2_1024_mixed_original: device=cuda, amp=bf16, seed=42
[09:30:16] Подготовка метаданных и train/val разбиения
[09:30:17] Метаданные загружены из кеша: D:\Challenges\AIIJC2026\data\train_stage1\.cache\metadata_v1.parquet
[09:30:17] Метаданные готовы: 103699 строк; построение 5 фолдов, val fold=0
[09:30:26] Разбиение готово: train=82959, val=20740; создание датасетов и аугментаций
[09:30:26] Создание модели efficientvit_b2.r288_in1k, загрузка pretrained-весов и перенос на cuda


model.safetensors:   0%|          | 0.00/97.5M [00:00<?, ?B/s]

D:\Apps\anaconda3\envs\challenges\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Юрий\.cache\huggingface\hub\models--timm--efficientvit_b2.r288_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[09:30:35] Модель готова; подсчёт GFLOPS
[09:30:41] Подсчёт завершён: 97.01 GFLOPS; создание оптимизатора
[09:30:41] Создание DataLoader: batch_size=2, workers=10
[09:30:41] DataLoader готовы: train=12000 батчей, val=5185; настройка scheduler, AMP scaler и EMA
[09:30:41] Создание папки эксперимента и сохранение конфигурации
[09:30:42] efficientvit_b2_1024_mixed_original, 97.00815744 GFLOPS
[09:30:42] Результаты: D:\Projects\aiijc2026_final\runs\efficientvit_b2_1024_mixed_original; проверка возобновления обучения
[09:30:42] Полные кадры: p=0.50; валидация: original
[09:30:42] Эпоха 1/8: обучение
[09:30:42] Обучение, батчи: начало, всего 12000; ожидание первого элемента
[09:31:36] Обучение, батчи: 1/12000, прошло 00:00:54
[09:32:06] Обучение, батчи: 175/12000, прошло 00:01:24, осталось ~00:34:08
[09:32:36] Обучение, батчи: 351/12000, прошло 00:01:54, осталось ~00:33:24
[09:33:06] Обучение, батчи: 526/12000, прошло 00:02:24, осталось ~00:32:55
[09:33:36] Обучение, батчи: 695/12000, прошло